In [40]:
import pandas as pd

In [50]:
# Loading the list customers

def load_customers(data_dir: str) -> pd.DataFrame:
    customers = pd.read_csv(f"{data_dir}/olist_customers_dataset.csv")
    orders = pd.read_csv(f"{data_dir}/olist_orders_dataset.csv")

    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

    merged = customers.merge(orders, on = "customer_id")

    result = merged.groupby("customer_unique_id").agg(
        first_order_date = ("order_purchase_timestamp", "min"),
        last_order_date = ("order_purchase_timestamp", "max"),
    ).reset_index()

    return result

In [31]:
# Checking the dataset

customers_df = load_customers("../data/raw")  # adjust path to your CSVs
print(customers_df.shape)
customers_df.head()

In [51]:
# Generating sessions (# of sessions + device_type per user)

import numpy as np
import uuid
from datetime import timedelta

def generate_sessions(customers_df):
    rows = []
    for _, customer in customers_df.iterrows():
        num_sessions = np.random.randint(1, 6)
        for _ in range(num_sessions):
            rows.append({
                "customer_unique_id": customer["customer_unique_id"],
                "session_id": str(uuid.uuid4()),
                "session_start": customer["first_order_date"] - timedelta(seconds = np.random.randint(0, 30 * 24 * 60 * 60)),
                "device_type": np.random.choice(["mobile", "desktop", "tablet"])
            })
    sessions_df = pd.DataFrame(rows)
    return sessions_df

In [52]:
# Generate AB assignments
def generate_ab_assignments(customers_df):
    rows = []
    experiments = ["checkout_redesign", "discount_campaign"]
    variants = ["control", "treatment"]
    for _, customer in customers_df.iterrows():
        assignment_time = customer["first_order_date"] - timedelta(minutes = np.random.randint(1, 10))
        temp = []
        for experiment in experiments:
            rows.append({
                "customer_unique_id": customer["customer_unique_id"],
                "experiment_id": experiment,
                "variant": np.random.choice(["control", "treatment"]),
                "assignment_timestamp": assignment_time
            })
    assignment_df = pd.DataFrame(rows)
    return assignment_df


In [57]:
def generate_funnel_events(sessions_df, ab_df):
    rows = []
    funnel_steps = ["page_view", "product_view", "add_to_cart", "checkout", "purchase"]

    sessions_with_variant = sessions_df.merge(
        ab_df[ab_df["experiment_id"] == "checkout_redesign"][["customer_unique_id", "variant"]],
        on="customer_unique_id"
    )
    sessions_with_variant["drop_off_prob"] = sessions_with_variant["variant"].map({
        "treatment": 0.27, # simulate 3% conversion lift in treatment
        "control": 0.30
    })

    for _, session in sessions_with_variant.iterrows():
        events = []
        current_time = session["session_start"]
        drop_off_prob = session["drop_off_prob"]
        for step in funnel_steps:
            current_time = current_time + timedelta(minutes=np.random.randint(1, 10))
            events.append((step, current_time))
            if np.random.random() < drop_off_prob:
                break
        for step, timestamp in events:
            rows.append({
                "session_id": session["session_id"],
                "customer_unique_id": session["customer_unique_id"],
                "event_type": step,
                "event_timestamp": timestamp
            })
    events_df = pd.DataFrame(rows)
    return events_df


In [58]:
import os
os.makedirs("../data/synthetic", exist_ok=True)

def main():
    customers_df = load_customers("../data/raw")
    customers_df.to_csv("../data/synthetic/customers.csv", index=False)
    print(f"Customers saved: {len(customers_df)} rows")

    sessions_df = generate_sessions(customers_df)
    sessions_df.to_csv("../data/synthetic/sessions.csv", index=False)
    print(f"Sessions saved: {len(sessions_df)} rows")

    ab_df = generate_ab_assignments(customers_df)
    ab_df.to_csv("../data/synthetic/ab_assignments.csv", index=False)
    print(f"AB assignments saved: {len(ab_df)} rows")

    events_df = generate_funnel_events(sessions_df, ab_df)
    events_df.to_csv("../data/synthetic/funnel_events.csv", index=False)
    print(f"Events saved: {len(events_df)} rows")

main()


Customers saved: 96096 rows
Sessions saved: 288328 rows
AB assignments saved: 192192 rows
Events saved: 823618 rows


In [67]:
# Final check
events_df = pd.read_csv("../data/synthetic/funnel_events.csv")
ab_df = pd.read_csv("../data/synthetic/ab_assignments.csv")

print(events_df.shape)
print(events_df["event_type"].value_counts())

purchases = events_df[events_df["event_type"] == "purchase"].merge(
    ab_df[ab_df["experiment_id"] == "checkout_redesign"],
    on="customer_unique_id"
)
print(purchases["variant"].value_counts())

(823618, 4)
event_type
page_view       288328
product_view    206253
add_to_cart     147645
checkout        105769
purchase         75623
Name: count, dtype: int64
variant
treatment    40944
control      34679
Name: count, dtype: int64
